<a href="https://colab.research.google.com/github/chiraswykj/GENAI_TUTORIALS/blob/main/MULTIAGENT_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq langgraph

In [ ]:
from groq import Groq
from google.colab import userdata

# Retrieve API key from Colab Secrets
api_key = userdata.get("GROQ_API_KEY")

# Initialize Groq client
client = Groq(api_key=api_key)

print("Groq API connected successfully!")

Groq API connected successfully!


In [ ]:
def create_agent(role, task):

    prompt = f"""
    You are an AI agent.

    Your role: {role}

    Your responsibility:
    {task}

    Provide a clear, detailed and structured response.
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": "Perform your assigned task."}
        ],
        temperature=0.5
    )

    return response.choices[0].message.content

In [ ]:
def planner_agent(user_task):

    return create_agent(
        role="Project Planner",
        task=f"""
        Understand the following user task:
        {user_task}

        Break it into clear steps.
        Identify the main objectives.
        Suggest how to solve the task.
        """
    )

In [ ]:
def researcher_agent(plan):

    return create_agent(
        role="Research Specialist",
        task=f"""
        Research the following plan:

        {plan}

        Explain relevant concepts,
        provide useful information,
        and identify important details.
        """
    )

In [ ]:
def analyst_agent(research):

    return create_agent(
        role="Data Analyst",
        task=f"""
        Analyse the following research:

        {research}

        Identify key findings,
        advantages, disadvantages,
        possible problems and recommendations.
        """
    )

In [ ]:
def writer_agent(analysis):

    return create_agent(
        role="Professional Report Writer",
        task=f"""
        Create a final report using this analysis:

        {analysis}

        Include:
        1. Introduction
        2. Main findings
        3. Analysis
        4. Recommendations
        5. Conclusion

        Make the report clear and well structured.
        """
    )

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph imported successfully!")

LangGraph imported successfully!


In [ ]:
class AgentState(TypedDict):

    user_task: str
    plan: str
    research: str
    analysis: str
    final_report: str

In [ ]:
def planner_node(state: AgentState):

    result = planner_agent(state["user_task"])

    return {"plan": result}


def researcher_node(state: AgentState):

    result = researcher_agent(state["plan"])

    return {"research": result}


def analyst_node(state: AgentState):

    result = analyst_agent(state["research"])

    return {"analysis": result}


def writer_node(state: AgentState):

    result = writer_agent(state["analysis"])

    return {"final_report": result}

In [ ]:
workflow = StateGraph(AgentState)

# Add agents as nodes
workflow.add_node("planner", planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("analyst", analyst_node)
workflow.add_node("writer", writer_node)

# Define execution order
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "researcher")
workflow.add_edge("researcher", "analyst")
workflow.add_edge("analyst", "writer")
workflow.add_edge("writer", END)

# Compile workflow
app = workflow.compile()

print("Multi-Agent Workflow created successfully!")

Multi-Agent Workflow created successfully!


In [ ]:
user_input = """
Explain how Artificial Intelligence
can improve the healthcare industry.
Include benefits, challenges and future opportunities.
"""

initial_state = {
    "user_task": user_input,
    "plan": "",
    "research": "",
    "analysis": "",
    "final_report": ""
}

result = app.invoke(initial_state)

print("Multi-Agent Workflow Completed!")

Multi-Agent Workflow Completed!


In [ ]:
print("\n========== PLANNER OUTPUT ==========\n")
print(result["plan"])

print("\n========== RESEARCHER OUTPUT ==========\n")
print(result["research"])

print("\n========== ANALYST OUTPUT ==========\n")
print(result["analysis"])

print("\n========== FINAL REPORT ==========\n")
print(result["final_report"])


========== PLANNER OUTPUT ==========

## Project Plan – “How Artificial Intelligence Can Improve the Healthcare Industry”  

Below is a **complete, step‑by‑step plan** that will guide you (or a writing team) from a blank page to a polished, well‑structured explanation that covers **benefits, challenges, and future opportunities** of AI in healthcare.  

---

### 1.  Main Objectives  

| # | Objective | Success Metric |
|---|-----------|----------------|
| 1 | **Explain the current role of AI in healthcare** (diagnostics, treatment, operations, research). | Clear, concise introductory paragraph (≈150‑200 words). |
| 2 | **Identify and describe the key benefits** (clinical, economic, patient‑centred). | ≥ 5 distinct benefit categories, each with a concrete example and supporting data point. |
| 3 | **Outline the major challenges and risks** (technical, ethical, regulatory, workforce). | ≥ 4 challenge categories, each paired with mitigation strategies. |
| 4 | **Highlight future opportun

In [ ]:
user_task = input("Enter your task: ")

initial_state = {
    "user_task": user_task,
    "plan": "",
    "research": "",
    "analysis": "",
    "final_report": ""
}

result = app.invoke(initial_state)

print("\n========== FINAL AI REPORT ==========\n")
print(result["final_report"])

Enter your task: Explain how renewable energy can help India achieve sustainable development.

========== FINAL AI REPORT ==========

**Policy Brief – Renewable‑Energy Deployment & the Sustainable Development Goals in India**  
*Prepared for: Policy‑makers, NGOs, investors and academia*  
*Date: 18 Nov 2026*  

---

## 1. Introduction  

India’s renewable‑energy (RE) sector has entered a decisive growth phase. By the end of FY‑2024, roughly **170 GW** of RE capacity—about **45 % of the nation’s total generation capacity**—was operating, delivering **38 % of electricity generation** (≈ 530 TWh). This rapid expansion is driven by ambitious policy instruments (RPO 30 % by 2030, competitive FIT auctions, Green Energy Corridors) and an expanding pool of green finance (≈ US$ 5 bn in green‑bond issuance, US$ 2 bn in blended financing).  

The purpose of this report is to synthesize the latest evidence (up to October 2024) and to show how scaling RE can **accelerate progress on all 17 Sustaina

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def run_multi_agent(user_task):

    if not user_task.strip():
        return "Please enter a task."

    initial_state = {
        "user_task": user_task,
        "plan": "",
        "research": "",
        "analysis": "",
        "final_report": ""
    }

    result = app.invoke(initial_state)

    return f"""
    # Final AI Report

    {result['final_report']}
    """

In [ ]:
# Define the missing model configuration
MODEL = "llama3-8b-8192"

# Re-build and compile the LangGraph workflow to ensure 'app' is defined
workflow = StateGraph(AgentState)

# Add agents as nodes
workflow.add_node("planner", planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("analyst", analyst_node)
workflow.add_node("writer", writer_node)

# Define execution order
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "researcher")
workflow.add_edge("researcher", "analyst")
workflow.add_edge("analyst", "writer")
workflow.add_edge("writer", END)

# Compile workflow
app = workflow.compile()

print("Model configured and workflow successfully rebuilt!")

In [ ]:
import gradio as gr
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Ensure AgentState is defined in this scope
class AgentState(TypedDict):
    user_task: str
    plan: str
    research: str
    analysis: str
    final_report: str

# Ensure MODEL and agent node functions are defined
MODEL = "llama3-8b-8192"

def planner_node(state: AgentState):
    result = planner_agent(state["user_task"])
    return {"plan": result}

def researcher_node(state: AgentState):
    result = researcher_agent(state["plan"])
    return {"research": result}

def analyst_node(state: AgentState):
    result = analyst_agent(state["research"])
    return {"analysis": result}

def writer_node(state: AgentState):
    result = writer_agent(state["analysis"])
    return {"final_report": result}

workflow = StateGraph(AgentState)
workflow.add_node("planner", planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("analyst", analyst_node)
workflow.add_node("writer", writer_node)

workflow.add_edge(START, "planner")
workflow.add_edge("planner", "researcher")
workflow.add_edge("researcher", "analyst")
workflow.add_edge("analyst", "writer")
workflow.add_edge("writer", END)

app = workflow.compile()

def run_multi_agent(user_task):
    if not user_task.strip():
        return "Please enter a task."

    initial_state = {
        "user_task": user_task,
        "plan": "",
        "research": "",
        "analysis": "",
        "final_report": ""
    }

    result = app.invoke(initial_state)
    return f"# Final AI Report\n\n{result['final_report']}"

with gr.Blocks() as demo:
    gr.Markdown("""
    # 🤖 Multi-Agent AI Assistant

    ### Your intelligent AI problem-solving system

    Enter a task and let four AI agents collaborate.
    """)

    user_input = gr.Textbox(
        label="Enter Your Task",
        placeholder="Example: Explain how AI is transforming healthcare",
        lines=4
    )

    submit_btn = gr.Button(
        "Generate Report",
        variant="primary"
    )

    output = gr.Markdown(
        label="AI Generated Report"
    )

    submit_btn.click(
        fn=run_multi_agent,
        inputs=user_input,
        outputs=output
    )

demo.launch(theme=gr.themes.Soft(), share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c6f19e76743f3fb168.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

<div class="markdown-google-sans">

## Machine learning
</div>

With Colab you can import an image dataset, train an image classifier on it, and evaluate the model, all in just [a few lines of code](https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb).

Colab is used extensively in the machine learning community with applications including:
- Getting started with TensorFlow
- Developing and training neural networks
- Experimenting with TPUs
- Disseminating AI research
- Creating tutorials

To see sample Colab notebooks that demonstrate machine learning applications, see the [machine learning examples](#machine-learning-examples) below.

<div class="markdown-google-sans">

## More Resources

### Working with Notebooks in Colab

</div>

- [Overview of Colab](/notebooks/basic_features_overview.ipynb)
- [Guide to Markdown](/notebooks/markdown_guide.ipynb)
- [Importing libraries and installing dependencies](/notebooks/snippets/importing_libraries.ipynb)
- [Saving and loading notebooks in GitHub](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/colab-github-demo.ipynb)
- [Interactive forms](/notebooks/forms.ipynb)
- [Interactive widgets](/notebooks/widgets.ipynb)

<div class="markdown-google-sans">

<a name="working-with-data"></a>
### Working with Data
</div>

- [Loading data: Drive, Sheets, and Google Cloud Storage](/notebooks/io.ipynb)
- [Charts: visualizing data](/notebooks/charts.ipynb)
- [Getting started with BigQuery](/notebooks/bigquery.ipynb)

<div class="markdown-google-sans">

### Machine Learning

<div>

These are a few of the notebooks related to Machine Learning, including Google's online Machine Learning course. See the [full course website](https://developers.google.com/machine-learning/crash-course/) for more.
- [Intro to Pandas DataFrame](https://colab.research.google.com/github/google/eng-edu/blob/main/ml/cc/exercises/pandas_dataframe_ultraquick_tutorial.ipynb)
- [Intro to RAPIDS cuDF to accelerate pandas](https://nvda.ws/rapids-cudf)
- [Getting Started with cuML's accelerator mode](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/getting_started_tutorials/cuml_sklearn_colab_demo.ipynb)

<div class="markdown-google-sans">

<a name="using-accelerated-hardware"></a>
### Using Accelerated Hardware
</div>

- [Train a CNN to classify handwritten digits on the MNIST dataset using Flax NNX API](https://colab.research.google.com/github/google/flax/blob/main/docs_nnx/mnist_tutorial.ipynb)
- [Train a Vision Transformer (ViT) for image classification with JAX](https://colab.research.google.com/github/jax-ml/jax-ai-stack/blob/main/docs/source/JAX_Vision_transformer.ipynb)
- [Text classification with a transformer language model using JAX](https://colab.research.google.com/github/jax-ml/jax-ai-stack/blob/main/docs/source/JAX_transformer_text_classification.ipynb)

<div class="markdown-google-sans">

<a name="machine-learning-examples"></a>

### Featured examples

</div>

- [Train a miniGPT language model with JAX AI Stack](https://docs.jaxstack.ai/en/latest/JAX_for_LLM_pretraining.html)
- [LoRA/QLoRA finetuning for LLM using Tunix](https://github.com/google/tunix/blob/main/examples/qlora_gemma.ipynb)
- [Parameter-efficient fine-tuning of Gemma with LoRA and QLoRA](https://keras.io/examples/keras_recipes/parameter_efficient_finetuning_of_gemma_with_lora_and_qlora/)
- [Loading Hugging Face Transformers Checkpoints](https://keras.io/keras_hub/guides/hugging_face_keras_integration/)
- [8-bit Integer Quantization in Keras](https://keras.io/guides/int8_quantization_in_keras/)
- [Float8 training and inference with a simple Transformer model](https://keras.io/examples/keras_recipes/float8_training_and_inference_with_transformer/)
- [Pretraining a Transformer from scratch with KerasHub](https://keras.io/keras_hub/guides/transformer_pretraining/)
- [Simple MNIST convnet](https://keras.io/examples/vision/mnist_convnet/)
- [Image classification from scratch using Keras 3](https://keras.io/examples/vision/image_classification_from_scratch/)
- [Image Classification with KerasHub](https://keras.io/keras_hub/guides/classification_with_keras_hub/)
